# 01 — End-to-end skeleton (dummy baseline)

Goal: prove that the shared machinery works end to end. Nothing here is a good
model.

If `data/dataset_train.csv` exists, the notebook uses it; otherwise it falls
back to synthetic data. Every saved run goes to `outputs/runs/` and appends a
row to `outputs/leaderboard.csv`.


In [ ]:
from pathlib import Path

import pandas as pd

from daa.config import OUTPUTS_DIR, TRAIN_PATH
from daa.data import TARGET, infer_column_spec, load_train
from daa.experiment import run_experiment
from daa.splits import SplitConfig, get_folds, make_folds


In [ ]:
if TRAIN_PATH.exists():
    X, y = load_train()
    folds = get_folds(y)
    print(f"real data: {X.shape[0]:,} rows x {X.shape[1]:,} columns")
else:
    from daa.fixtures import make_synthetic_frame

    frame = make_synthetic_frame(n_samples=240, seed=0)
    X = frame.drop(columns=[TARGET])
    y = frame[TARGET].astype(int)
    folds = make_folds(y, SplitConfig(n_splits=3, n_repeats=1, seed=0))
    print(f"synthetic data: {X.shape[0]:,} rows x {X.shape[1]:,} columns")

print(f"{len(folds)} folds, {y.nunique()} classes")


In [ ]:
spec = infer_column_spec(X)
spec.counts()


## The dumbest possible baseline

`DummyClassifier(strategy="most_frequent")` always predicts the majority class.
It is the floor: any real model must beat it on macro F1.


In [ ]:
dummy_result = run_experiment(
    "dummy",
    name="dummy_baseline",
    X=X,
    y=y,
    folds=folds,
    fit_full=True,
)
pd.Series(dummy_result.metrics, name="dummy")


In [ ]:
dummy_result.per_class


## A first real baseline

A shallow decision tree. It is fast, handles mixed data after preprocessing,
and gives us a reference before anyone tunes anything.


In [ ]:
tree_result = run_experiment(
    "decision_tree",
    {"max_depth": 12, "min_samples_leaf": 20},
    name="decision_tree_baseline",
    X=X,
    y=y,
    folds=folds,
    fit_full=True,
)
pd.Series(tree_result.metrics, name="decision_tree")


In [ ]:
tree_result.per_class


In [ ]:
tree_result.confusion


## Where the run went

The leaderboard is the shared comparison surface. A, B and C all append to it,
and nobody edits anyone else's row.


In [ ]:
print(OUTPUTS_DIR / "leaderboard.csv")
pd.read_csv(OUTPUTS_DIR / "leaderboard.csv").tail()


## Next

- **A** adds feature blocks in `src/daa/features/` and compares variants.
- **B** adds/tunes models in `src/daa/models/`.
- **C** builds the report, error analysis, and the submission from the stored
  run directories.

All three use the same folds and the same metrics, so the leaderboard stays
comparable.
